# Fase 3B — DistilBETO (IroSvA)

**Nested CV 5×3 agrupada · Optuna TPE (20 trials) · F1-Macro**

Versión alineada con BETO y RoBERTuito: mismo preprocesamiento, mismo protocolo y mismo espacio de búsqueda.

DistilBETO recibe el train oficial preprocesado por `01_preprocesamiento_dl_transformers.ipynb` (sin retirar registros en función del test) y se limita a:

- verificar el manifest;
- construir sus `GROUP_ID` propios;
- ejecutar `StratifiedGroupKFold` 5×3;
- optimizar con F1-Macro;
- guardar checkpoints, curvas y resultados por Outer Fold.

No elimina filas utilizando el test oficial.

Los tres Transformers reciben **el mismo texto**, `MESSAGE_TRANSFORMER`, generado con el preprocesamiento oficial de Pérez et al. (2022) (`pysentimiento.preprocessing.preprocess_tweet`).

### Revisión antes de la corrida larga

Esta versión mantiene las correcciones de robustez ya incorporadas:

1. al reanudar un checkpoint se restaura también el estado de callbacks, incluido `EarlyStoppingCallback`;
2. los trials interrumpidos de Optuna se cierran mediante la API pública `Study.tell(..., FAIL)`;
3. la historia de entrenamiento se congela antes de evaluaciones posteriores para no alterar curvas ni `best_epoch`;
4. el preprocesamiento no retira registros del train en función del test; las coincidencias train ↔ test se reportan solo en la evaluación final, como análisis de sensibilidad;
5. BETO, DistilBETO y RoBERTuito verifican el mismo manifest y reciben el mismo texto, pero construyen grupos y folds propios;
6. si `MESSAGE_TRANSFORMER` está vacío (preprocesamiento ejecutado sin `pysentimiento`), la carga se detiene.

### Estructura de checkpoints

Todos los Transformers usan la misma organización compacta para Windows:

```text
seguimiento/
└── d5x3t20_v3/
    └── completo/
        ├── ckpt/
        ├── state/
        ├── curves/
        ├── graphs/
        ├── audit/
        └── tmp/
```

Un ejemplo de Inner Fold queda:

```text
ckpt/mx/outer1/P_l5_b16_e5_w5_s_d3/inner1/checkpoint-...
```

Codificación de la configuración:

- `P`: texto común `MESSAGE_TRANSFORMER` (preprocesamiento de pysentimiento)
- `l5`: learning rate `5e-5`
- `b16`: batch size 16
- `e5`: máximo 5 épocas
- `w5`: weight decay `0.05`
- `s` / `b`: standard / balanced loss
- `d1` / `d2` / `d3`: `classifier_dropout=0.1/0.2/0.3`

Los archivos `state` y `curves` quedan fuera de la carpeta pesada de checkpoints, de modo que los checkpoints ya completados pueden borrarse sin perder resultados ni curvas.

## 1. Librerías y entorno

En la terminal del proyecto (mismo entorno que usa Jupyter):

```powershell
py -m uv add torch transformers accelerate optuna scikit-learn pandas numpy ipython
```

Si ya tienes las dependencias, no necesitas reinstalarlas. Este notebook **no instala paquetes ni inicia GPU** por sí solo al abrirlo.

In [ ]:
import os
import re

# Declarar antes de importar Transformers/Hugging Face para ocultar barras de descarga.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

import gc
import time
import json
import pickle
import html
import random
import tempfile
import shutil
import inspect
import warnings
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import optuna

from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from IPython.display import display, HTML

import matplotlib

matplotlib.use("Agg")  # Evita abrir una ventana por gráfica durante cientos de fits.
import matplotlib.pyplot as plt

import transformers
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed,
)
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from transformers.trainer_utils import get_last_checkpoint

try:
    from huggingface_hub.utils import disable_progress_bars

    disable_progress_bars()
except (ImportError, AttributeError):
    pass
transformers.logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

VERSIONES = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "optuna": optuna.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 2. Configuración

In [ ]:
MODEL_NAME = "dccuchile/distilbert-base-spanish-uncased"

DATA_DIR = Path("../data/transformers")
RESULTS_DIR = Path("../data/resultados_distilbeto_grouped_5x3_tpe20_v3")
PROGRESS_DIR = RESULTS_DIR / "seguimiento"

TEXT_COLS = {"PYSENTIMIENTO": "MESSAGE_TRANSFORMER"}
LABEL_COL = "IS_IRONIC"
ID_COL = "ID"
ORIGINAL_COL = "MESSAGE"
GROUP_COL = "GROUP_ID"

MAX_LENGTH = 128
SEED = 42

# ------------------------------------------------------------
# "prueba" valida rápidamente todo el pipeline.
# "completo" = experimento definitivo 5×3 / TPE20.
# ------------------------------------------------------------
MODO = "completo"

if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER = 2
    N_INNER = 2
    N_TRIALS = 2
    N_STARTUP_TRIALS = 1
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER = 5
    N_INNER = 3
    N_TRIALS = 20
    N_STARTUP_TRIALS = 5
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

# RUN_TAG corto + subcarpetas compactas para Windows.
# Carpeta nueva: corrida con el preprocesamiento estandarizado
# (sin retirar registros del train en función del test).
RUN_TAG = "d5x3t20_v3"
RUN_DIR = PROGRESS_DIR / RUN_TAG / MODO
TMP_DIR = RUN_DIR / "tmp"
CHECKPOINT_DIR = RUN_DIR / "ckpt"
FIT_STATE_DIR = RUN_DIR / "state"
CURVE_DIR = RUN_DIR / "curves"
GRAPH_DIR = RUN_DIR / "graphs"
AUDIT_DIR = RUN_DIR / "audit"

for carpeta in [
    RESULTS_DIR,
    PROGRESS_DIR,
    RUN_DIR,
    TMP_DIR,
    CHECKPOINT_DIR,
    FIT_STATE_DIR,
    CURVE_DIR,
    GRAPH_DIR,
    AUDIT_DIR,
]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Flujo recomendado (sección 18): opcionalmente una corrida corta con MODO="prueba";
# luego MODO="completo". Nested CV y optimización final usan solo train y corren seguidas;
# el test oficial requiere además CONFIRMACION_TEST.
EJECUTAR_NESTED = True
EJECUTAR_OPTIMIZACION_FINAL = True  # solo train; se ejecuta tras la Nested CV
EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""

SEMILLAS_FINALES = [42, 52, 62, 72, 82]
CLEAN_COMPLETED_CHECKPOINTS = True

# Seguimiento: panel de progreso común a todos los notebooks de modelos.
NOMBRE_EXPERIMENTO = "DistilBETO"
HEARTBEAT_SEGUNDOS = 30
# Pitido de Windows al terminar o detenerse una etapa larga (False para desactivarlo).
AVISO_SONORO = True

# Optuna TPE: 5 trials iniciales aleatorios y 15 adaptativos en modo completo.
# Espacio común a BETO, DistilBETO y RoBERTuito.
SEARCH_SPACE = {
    "preprocessing": ["PYSENTIMIENTO"],
    "learning_rate": [2e-5, 3e-5, 4e-5, 5e-5],
    "batch_size": [16, 32],
    "num_train_epochs": [3, 4, 5],
    "weight_decay": [0.0, 0.01, 0.05, 0.1],
    "loss_mode": ["standard", "balanced"],
    "classifier_dropout": [0.1, 0.2, 0.3],
}

SEARCH_SPACE_REQUERIDO = {
    "preprocessing",
    "learning_rate",
    "batch_size",
    "num_train_epochs",
    "weight_decay",
    "loss_mode",
    "classifier_dropout",
}
faltantes_search_space = SEARCH_SPACE_REQUERIDO - set(SEARCH_SPACE)
if faltantes_search_space:
    raise KeyError(f"SEARCH_SPACE incompleto. Faltan: {sorted(faltantes_search_space)}")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

fits_teoricos = len(VARIANTES) * N_OUTER * (N_TRIALS * N_INNER + 1)

print(f"[CONFIG][DistilBETO] modo={MODO} | variantes={VARIANTES}")
print(
    f"[CONFIG][DistilBETO] Nested={N_OUTER}×{N_INNER} | "
    f"trials={N_TRIALS} | startup={N_STARTUP_TRIALS}"
)
print(
    f"[CONFIG][DistilBETO] fine-tunings teóricos={fits_teoricos} | "
    f"nested_activo={EJECUTAR_NESTED}"
)
print(f"[CONFIG][DistilBETO] RUN_DIR={RUN_DIR.resolve()}")
print("[CONFIG][DistilBETO] SEARCH_SPACE OK:", sorted(SEARCH_SPACE))

# Comprobación preventiva de longitud de ruta en Windows (peor caso del espacio).
_ruta_ckpt_ejemplo = (
    CHECKPOINT_DIR / "mx" / "outer1" / "P_l5_b32_e5_w5_b_d3" / "inner3" / "checkpoint-99999"
).resolve()

print(
    f"[CONFIG][DistilBETO] longitud ruta checkpoint ejemplo="
    f"{len(str(_ruta_ckpt_ejemplo))} caracteres"
)

if os.name == "nt" and len(str(_ruta_ckpt_ejemplo)) >= 230:
    raise RuntimeError(
        "La ruta prevista para checkpoints sigue siendo demasiado larga en Windows. "
        "Mueve el repositorio a una ruta más corta antes de entrenar."
    )

## 3. Carga del train y verificación del preprocesamiento

BETO, DistilBETO y RoBERTuito consumen los mismos CSV finales.

Antes de construir grupos se verifica:

- manifest del preprocesamiento y versión del pipeline;
- número final de registros por variante;
- IDs únicos;
- etiquetas binarias;
- `MESSAGE_TRANSFORMER` no vacío.

Este notebook no retira registros del train en función del test.

In [ ]:
MANIFEST_PATH = DATA_DIR / "manifest_preprocesamiento_transformers.csv"
PIPELINE_VERSION_ESPERADA = "transformers_comun_v2"


def cargar_manifest_preprocesamiento():
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Falta {MANIFEST_PATH}. Ejecuta primero "
            "`01_preprocesamiento_dl_transformers.ipynb`."
        )

    manifest = pd.read_csv(MANIFEST_PATH)

    requeridas = {"pipeline_version", "variante", "train_final"}
    faltantes = requeridas - set(manifest.columns)
    if faltantes:
        raise ValueError(f"Manifest incompleto; faltan columnas: {sorted(faltantes)}")

    versiones = set(manifest["pipeline_version"].astype(str))
    if versiones != {PIPELINE_VERSION_ESPERADA}:
        raise ValueError(
            f"Versión de preprocessing inesperada: {versiones}. "
            f"Se esperaba {PIPELINE_VERSION_ESPERADA}."
        )

    return manifest


MANIFEST_PREPROCESAMIENTO = cargar_manifest_preprocesamiento()


def validar_archivo_train_con_manifest(variante):
    ruta = DATA_DIR / f"train_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}. Ejecuta primero el preprocessing Transformer.")

    fila = MANIFEST_PREPROCESAMIENTO[MANIFEST_PREPROCESAMIENTO["variante"].astype(str) == variante]
    if len(fila) != 1:
        raise ValueError(f"{variante}: se esperaba exactamente una fila en el manifest.")

    return ruta, fila.iloc[0]


def cargar_train(variante):
    ruta, manifest_row = validar_archivo_train_con_manifest(variante)
    df = pd.read_csv(ruta)

    requeridas = [ID_COL, ORIGINAL_COL, LABEL_COL, *TEXT_COLS.values()]
    faltantes = [c for c in requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f"{variante}: faltan columnas {faltantes}")

    df[ID_COL] = df[ID_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    for col in [ORIGINAL_COL, *TEXT_COLS.values()]:
        df[col] = df[col].fillna("").astype(str)

    if df[ID_COL].duplicated().any():
        raise ValueError(f"{variante}: IDs duplicados en el train FINAL.")

    if not set(df[LABEL_COL].unique()).issubset({0, 1}):
        raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

    vacios = {
        col: int(df[col].eq("").sum())
        for col in [ORIGINAL_COL, *TEXT_COLS.values()]
        if df[col].eq("").any()
    }
    if vacios:
        raise ValueError(
            f"{variante}: entradas vacías en el train final {vacios}. "
            "Vuelve a ejecutar `01_preprocesamiento_dl_transformers.ipynb` "
            "(MESSAGE_TRANSFORMER requiere `pysentimiento` instalado)."
        )

    esperado = int(manifest_row["train_final"])
    if len(df) != esperado:
        raise ValueError(f"{variante}: train={len(df)} pero manifest={esperado}.")

    return df.reset_index(drop=True)


for variante in ["mx", "es", "cu"]:
    tmp = cargar_train(variante)
    print(
        f"[DATA][DistilBETO][{variante.upper()}] "
        f"n={len(tmp)} | "
        f"clases={tmp[LABEL_COL].value_counts().sort_index().to_dict()} | "
        "manifest=OK"
    )

## 4. Tokenizador oficial y datos

Se usa el **tokenizer propio de DistilBETO** (uncased: el propio tokenizer pasa el texto a minúsculas). Los tres Transformers reciben el mismo texto, `MESSAGE_TRANSFORMER`, generado con `pysentimiento.preprocessing.preprocess_tweet`. No se elimina puntuación, tildes ni stopwords. El texto se tokeniza previamente (sin usar etiquetas para crear el vocabulario) y cada partición utiliza índices del mismo dataframe. Padding dinámico por batch.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer:", tokenizer.__class__.__name__)
print("do_lower_case:", getattr(tokenizer, "do_lower_case", "gestionado por tokenizer"))
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None
)


class DatasetTokenizado(Dataset):
    def __init__(self, textos, etiquetas):
        self.encoded = tokenizer(
            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False
        )
        self.labels = np.asarray(etiquetas, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encoded.items()}
        item["labels"] = int(self.labels[i])
        return item


def cache_dataset(df):
    y = df[LABEL_COL].to_numpy(dtype=np.int64)
    return {prep: DatasetTokenizado(df[col].tolist(), y) for prep, col in TEXT_COLS.items()}, y

### 4.1 Auditoría y construcción de `GROUP_ID`

DistilBETO crea sus grupos **únicamente para impedir leakage entre folds** dentro del train.

Se consideran equivalentes:

- texto original;
- `MESSAGE_TRANSFORMER`;
- secuencia tokenizada de `MESSAGE_TRANSFORMER` con el tokenizer de DistilBETO.

Los grupos son propios de DistilBETO y no dependen de los grupos o splits de los otros modelos.

In [ ]:
from collections import defaultdict


def colapsar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto)).strip()


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def clave_tokens(ids):
    """Representación directa de los input_ids, sin hashes."""
    return ",".join(map(str, ids))


def construir_grupos_distilbeto(df, variante, tokenizador=tokenizer, max_length=MAX_LENGTH):
    """Agrupa entradas equivalentes para evitar que crucen train/validación.

    Agrupación propia del modelo: texto original, cada representación de
    entrada (TEXT_COLS) y su secuencia tokenizada (truncada a MAX_LENGTH).
    """
    trabajo = df.copy().reset_index(drop=True)
    trabajo["KEY_ORIGINAL"] = trabajo[ORIGINAL_COL].map(colapsar_espacios)
    for prep, columna in TEXT_COLS.items():
        trabajo[f"KEY_{prep}"] = trabajo[columna].map(colapsar_espacios)

    for prep, columna in TEXT_COLS.items():
        ids = tokenizador(
            trabajo[columna].astype(str).tolist(),
            truncation=True,
            max_length=max_length,
            padding=False,
        )["input_ids"]
        trabajo[f"KEY_TOK_{prep}"] = [clave_tokens(x) for x in ids]

    representaciones = ["ORIGINAL", *TEXT_COLS, *[f"TOK_{p}" for p in TEXT_COLS]]
    key_cols = [f"KEY_{r}" for r in representaciones]
    uf = UnionFind(len(trabajo))

    for col in key_cols:
        first_seen = {}
        for idx, key in enumerate(trabajo[col].tolist()):
            if not key:
                continue
            if key in first_seen:
                uf.union(idx, first_seen[key])
            else:
                first_seen[key] = idx

    componentes = defaultdict(list)
    for idx in range(len(trabajo)):
        componentes[uf.find(idx)].append(idx)

    group_ids = [None] * len(trabajo)

    componentes_ordenados = sorted(componentes.values(), key=lambda miembros: min(miembros))

    for numero_grupo, miembros in enumerate(componentes_ordenados, start=1):
        gid = f"{variante}_G{numero_grupo:04d}"
        for idx in miembros:
            group_ids[idx] = gid
    trabajo[GROUP_COL] = group_ids

    resumen = []
    for nombre, col in zip(representaciones, key_cols):
        sizes = trabajo.groupby(col).size()
        repetidos = sizes[sizes > 1]
        resumen.append(
            {
                "variante": variante,
                "representacion": nombre,
                "filas_en_repeticiones": int(trabajo[col].isin(repetidos.index).sum()),
                "grupos_repetidos": int(len(repetidos)),
            }
        )

    gstats = trabajo.groupby(GROUP_COL)[LABEL_COL].agg(["size", "nunique"])
    conflictivos = set(gstats[(gstats["size"] > 1) & (gstats["nunique"] > 1)].index)
    detalle_conflictos = trabajo[trabajo[GROUP_COL].isin(conflictivos)].copy()

    resumen = pd.DataFrame(resumen)
    resumen["n_train"] = len(trabajo)
    resumen["n_group_id"] = trabajo[GROUP_COL].nunique()
    resumen["max_group_size"] = int(trabajo.groupby(GROUP_COL).size().max())
    resumen["grupos_con_labels_mixtos"] = len(conflictivos)
    return trabajo, resumen, detalle_conflictos


TRAIN_AGRUPADO = {}
AUDITORIA_GRUPOS = []
DETALLES_CONFLICTOS = []

# Se construyen los grupos de las tres variantes antes de una corrida larga.
for variante in ["mx", "es", "cu"]:
    df_audit = cargar_train(variante)
    agrupado, resumen, conflictos = construir_grupos_distilbeto(df_audit, variante)
    TRAIN_AGRUPADO[variante] = agrupado
    AUDITORIA_GRUPOS.append(resumen)
    if not conflictos.empty:
        conflictos["variante"] = variante
        DETALLES_CONFLICTOS.append(conflictos)

df_auditoria_grupos = pd.concat(AUDITORIA_GRUPOS, ignore_index=True)
df_auditoria_grupos.to_csv(RUN_DIR / "auditoria_grupos_train.csv", index=False)
display(df_auditoria_grupos)

if DETALLES_CONFLICTOS:
    df_conflictos_grupo = pd.concat(DETALLES_CONFLICTOS, ignore_index=True)
    df_conflictos_grupo.to_csv(RUN_DIR / "auditoria_grupos_labels_mixtos.csv", index=False)

for variante in ["mx", "es", "cu"]:
    df_g = TRAIN_AGRUPADO[variante]
    n_mixtos = int(df_g.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())
    print(
        f"[AUDIT][DistilBETO][{variante.upper()}] n={len(df_g)} | "
        f"groups={df_g[GROUP_COL].nunique()} | "
        f"max_group={df_g.groupby(GROUP_COL).size().max()} | labels_mixtos={n_mixtos}"
    )

    if n_mixtos > 0:
        print(
            f"  ADVERTENCIA: {n_mixtos} GROUP_ID contienen etiquetas mixtas. "
            "No es data leakage porque el grupo permanece íntegro dentro de un fold, "
            "pero representa ambigüedad/ruido de etiqueta y debe documentarse."
        )

### 4.2 Pre-flight metodológico

El pre-flight valida el train final y los grupos de DistilBETO.

No accede al test para eliminar o decidir registros.

In [ ]:
def comprobar_preflight_distilbeto():
    print("=" * 78)
    print("PRE-FLIGHT — DISTILBETO GROUPED NESTED CV")
    print("=" * 78)

    if MODO == "completo":
        esperado = (N_OUTER, N_INNER, N_TRIALS, N_STARTUP_TRIALS)

        if esperado != (5, 3, 20, 5):
            raise ValueError(
                f"Configuración inesperada: {esperado}; " "se esperaba 5×3, 20 trials, startup=5."
            )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA no disponible. Verifica entorno/PyTorch antes de entrenar DistilBETO."
        )

    for variante in ["mx", "es", "cu"]:
        _, manifest_row = validar_archivo_train_con_manifest(variante)

        df = TRAIN_AGRUPADO[variante]

        if len(df) != int(manifest_row["train_final"]):
            raise ValueError(f"{variante}: train agrupado no coincide " "con el manifest.")

        if df[ID_COL].duplicated().any():
            raise ValueError(f"{variante}: IDs duplicados.")

        if df[GROUP_COL].isna().any():
            raise ValueError(f"{variante}: GROUP_ID nulo.")

        if not set(df[LABEL_COL].unique()).issubset({0, 1}):
            raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

        n_mixtos = int(df.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())

        if n_mixtos:
            print(
                f"[AUDIT][DistilBETO][{variante.upper()}] "
                f"ADVERTENCIA: {n_mixtos} GROUP_ID "
                "con etiquetas mixtas; se documentan "
                "como ruido/ambigüedad."
            )

    print("Preprocessing:", PIPELINE_VERSION_ESPERADA)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Nested: {N_OUTER}×{N_INNER} | Optuna={N_TRIALS} | startup={N_STARTUP_TRIALS}")
    print("Train final:", {v: len(TRAIN_AGRUPADO[v]) for v in ["mx", "es", "cu"]})
    print("PRE-FLIGHT DE DATOS: OK")


comprobar_preflight_distilbeto()

## 5. Métricas, pesos de clase y modelos independientes

En cada Inner Fold la pérdida balanceada calcula pesos **solo** con el train de ese fold. Cada llamada `crear_modelo()` carga DistilBETO desde cero con nueva cabeza de clasificación binaria. Los mensajes de pesos `MISSING`/`UNEXPECTED` quedan en un log, no en Jupyter.

In [ ]:
def establecer_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def metricas(prediction):
    logits, labels = prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    y_pred = np.asarray(logits).argmax(axis=-1)
    return {
        "f1_macro": f1_score(labels, y_pred, average="macro"),
        "accuracy": accuracy_score(labels, y_pred),
        "precision_macro": precision_score(labels, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, y_pred, average="macro", zero_division=0),
    }


def pesos_balanceados(y):
    classes = np.array([0, 1])
    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y))
    return torch.tensor(w, dtype=torch.float32)


def crear_modelo(seed, classifier_dropout=0.2):
    establecer_semilla(seed)
    with (RUN_DIR / "carga_modelos.log").open("a", encoding="utf-8") as log, redirect_stdout(
        log
    ), redirect_stderr(log):
        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
        if config.model_type != "distilbert":
            raise ValueError(f"Se esperaba arquitectura DistilBERT, recibida: {config.model_type}")
        # DistilBERT guarda el dropout de la cabeza de clasificación en `seq_classif_dropout`.
        config.seq_classif_dropout = float(classifier_dropout)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    return model


class ClasificadorTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        self.class_weights = class_weights
        super().__init__(*args, **kwargs)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        if self.class_weights is None:
            loss = outputs.loss
        else:
            weights = self.class_weights.to(outputs.logits.device)
            loss = F.cross_entropy(outputs.logits, labels, weight=weights)
        return (loss, outputs) if return_outputs else loss

## 6. Argumentos de entrenamiento

El Trainer evalúa por época, guarda el mejor checkpoint por F1-Macro, utiliza AdamW + 10% warmup y detiene entrenamiento si F1 no mejora durante una evaluación. En el **refit externo y final no hay early stopping**, porque no se puede usar el Outer Test ni el test oficial como validación.

In [ ]:
def argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=True, epochs=None):
    params = {
        "output_dir": str(carpeta),
        "learning_rate": cfg["learning_rate"],
        "per_device_train_batch_size": cfg["batch_size"],
        "per_device_eval_batch_size": cfg["batch_size"],
        "num_train_epochs": epochs if epochs is not None else cfg["num_train_epochs"],
        "weight_decay": cfg["weight_decay"],
        "warmup_ratio": 0.10,
        "optim": "adamw_torch",
        # Trainer recoge loss por época en state.log_history;
        # PrinterCallback se elimina para no saturar VS Code.
        "logging_strategy": "epoch",
        "save_strategy": "epoch",  # Inner y refit: reinicio desde última época completa.
        "load_best_model_at_end": bool(con_validacion),
        "save_total_limit": 2,  # último + mejor (si son diferentes)
        "save_only_model": False,  # Estado optimizer/scheduler requerido para reanudar.
        "save_safetensors": True,
        "report_to": "none",
        "disable_tqdm": True,
        # Necesario para que EarlyStoppingCallback conserve su contador
        # al reanudar desde un checkpoint.
        "restore_callback_states_from_checkpoint": True,
        "dataloader_num_workers": 0,
        "seed": seed,
        "data_seed": seed,
        "bf16": USE_BF16,
        "fp16": USE_FP16,
    }
    if con_validacion:
        params.update({"metric_for_best_model": "f1_macro", "greater_is_better": True})
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        params["eval_strategy"] = "epoch" if con_validacion else "no"
    elif "evaluation_strategy" in sig:
        params["evaluation_strategy"] = "epoch" if con_validacion else "no"
    else:
        raise RuntimeError("Esta versión de Transformers no admite evaluación por época.")
    return TrainingArguments(**{k: v for k, v in params.items() if k in sig})


def mejor_epoca(trainer, max_epochs):
    evaluaciones = [x for x in trainer.state.log_history if "eval_f1_macro" in x and "epoch" in x]
    if not evaluaciones:
        raise RuntimeError("No se encontró F1 de validación por época.")
    mejor = max(evaluaciones, key=lambda x: x["eval_f1_macro"])
    return int(np.clip(round(float(mejor["epoch"])), 1, max_epochs))


def predecir(trainer, dataset):
    logits = trainer.predict(dataset).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    return np.argmax(logits, axis=-1)

## 7. Monitor único y archivos de recuperación

El panel se actualiza sin crear cientos de outputs. Al terminar una etapa (Nested CV, optimización final o test) el panel pasa a **verde con «✅ TERMINADO»**, la barra llega al 100 %, se imprime un mensaje con la hora en la celda y suena un aviso en Windows (`AVISO_SONORO`). Si se detiene por error o interrupción, el panel pasa a **rojo con «❌ DETENIDO»**. Se registran señales en `monitor.log` aproximadamente cada 30 segundos durante los pasos de entrenamiento. Los trials completos se guardan en una base SQLite por Outer Fold y los resultados de cada Outer Fold se escriben inmediatamente en CSV.

Si un entrenamiento se interrumpe, puedes ejecutar otra vez el notebook con **los mismos CSV y la misma configuración**. Se comprueba la configuración guardada para evitar mezclar experimentos distintos.

In [ ]:
class Monitor:
    """Panel único de progreso + monitor.log, con estado activo / terminado / error.

    Clase común a todos los notebooks de modelos (ML, Hard Voting y Transformers).
    """

    def __init__(self, ruta, total, ya_completados=0, unidad="Fine-tunings completados"):
        self.ruta = Path(ruta)
        self.ruta.mkdir(parents=True, exist_ok=True)
        self.log = self.ruta / "monitor.log"
        self.total = int(total)
        self.unidad = unidad
        self.completados = int(ya_completados)
        self.completados_inicio = int(ya_completados)
        self.inicio = time.monotonic()
        self.ultimo_pulso = 0
        self.etapa = "Inicializando"
        self.estado = "activo"  # activo | terminado | error
        self.handle = display(HTML(f"<b>Preparando {NOMBRE_EXPERIMENTO}...</b>"), display_id=True)
        self.mostrar(forzar=True)

    def registrar(self, msg):
        with self.log.open("a", encoding="utf-8") as f:
            f.write(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}\n")

    def mostrar(self, etapa=None, forzar=False):
        if etapa is not None:
            self.etapa = str(etapa)
        ahora = time.monotonic()
        if not forzar and ahora - self.ultimo_pulso < HEARTBEAT_SEGUNDOS:
            return
        self.ultimo_pulso = ahora
        minutos = (ahora - self.inicio) / 60
        hechos_sesion = self.completados - self.completados_inicio
        faltan = max(0, self.total - self.completados)
        eta = (
            f"~{minutos / hechos_sesion * faltan:.0f} min"
            if hechos_sesion >= 2
            else "calculando..."
        )
        marca = datetime.now().strftime("%H:%M:%S")
        color, titulo = {
            "activo": ("#777", f"{NOMBRE_EXPERIMENTO} — seguimiento en vivo"),
            "terminado": ("#2e7d32", f"✅ {NOMBRE_EXPERIMENTO} — TERMINADO"),
            "error": ("#c62828", f"❌ {NOMBRE_EXPERIMENTO} — DETENIDO"),
        }[self.estado]
        grosor = 1 if self.estado == "activo" else 3
        tiempo = (
            f"Tiempo: {minutos:.1f} min | ETA orientativo: {eta}"
            if self.estado == "activo"
            else f"Duración total: {minutos:.1f} min | Finalizó a las {marca}"
        )
        panel = (
            f'<div style="font-family:system-ui;padding:12px;border:{grosor}px solid {color};'
            'border-radius:8px;max-width:800px">'
            f'<b style="color:{color}">{titulo}</b><br>'
            f"Etapa: <b>{html.escape(self.etapa)}</b><br>"
            f"{self.unidad}: {self.completados}/{self.total} "
            f"({100*self.completados/max(1,self.total):.1f}%)<br>"
            f'<progress value="{self.completados}" max="{max(1,self.total)}" '
            'style="width:100%"></progress><br>'
            f"{tiempo}<br>"
            f"Última señal: {marca}<br>Log: {html.escape(str(self.log))}</div>"
        )
        self.handle.update(HTML(panel))

    def iniciar(self, msg):
        self.registrar(msg)
        self.mostrar(etapa=msg, forzar=True)

    def pulso(self):
        if time.monotonic() - self.ultimo_pulso >= HEARTBEAT_SEGUNDOS:
            self.registrar("ACTIVO: " + self.etapa)
            self.mostrar(forzar=True)

    def fit_terminado(self, msg):
        self.completados += 1
        self.registrar(msg)
        self.mostrar(etapa=msg, forzar=True)

    def finalizar(self, msg):
        """Etapa terminada: panel verde al 100 %, mensaje en la celda y aviso sonoro."""
        self.registrar(f"FINALIZADO: {msg} ({self.completados}/{self.total} contados)")
        self.estado = "terminado"
        self.completados = max(self.completados, self.total)
        self.mostrar(etapa=msg, forzar=True)
        print(f"\n✅ [{NOMBRE_EXPERIMENTO}] {msg} — {datetime.now():%Y-%m-%d %H:%M:%S}", flush=True)
        avisar_fin(ok=True)

    def fallar(self, msg):
        """Etapa detenida (error o interrupción): panel rojo, mensaje en la celda y aviso sonoro."""
        self.registrar(f"DETENIDO: {msg}")
        self.estado = "error"
        self.mostrar(etapa=msg, forzar=True)
        print(f"\n❌ [{NOMBRE_EXPERIMENTO}] {msg} — {datetime.now():%Y-%m-%d %H:%M:%S}", flush=True)
        avisar_fin(ok=False)


def avisar_fin(ok=True):
    """Pitido de Windows al terminar (3 veces) o al detenerse (1 vez); en otros sistemas no hace nada."""
    if not AVISO_SONORO:
        return
    try:
        import winsound

        for _ in range(3 if ok else 1):
            winsound.MessageBeep(winsound.MB_ICONASTERISK if ok else winsound.MB_ICONHAND)
            time.sleep(0.5)
    except Exception:
        pass


class PulsoTrainer(TrainerCallback):
    def __init__(self, monitor):
        self.monitor = monitor

    def on_step_end(self, args, state, control, **kwargs):
        self.monitor.pulso()
        return control


def entrenador_silencioso(trainer):
    trainer.remove_callback(PrinterCallback)
    trainer.remove_callback(ProgressCallback)
    return trainer


def csv_atomico(df, ruta):
    ruta = Path(ruta)
    tmp = ruta.with_name(ruta.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, ruta)


def configuracion_actual():
    return {
        "checkpoint": MODEL_NAME,
        "modo": MODO,
        "variantes": VARIANTES,
        "outer": N_OUTER,
        "inner": N_INNER,
        "trials": N_TRIALS,
        "startup_trials": N_STARTUP_TRIALS,
        "semilla": SEED,
        "max_length": MAX_LENGTH,
        "search_space": SEARCH_SPACE,
        "preprocessing_pipeline": PIPELINE_VERSION_ESPERADA,
        "train_final": {
            str(fila.variante): int(fila.train_final)
            for fila in MANIFEST_PREPROCESAMIENTO.itertuples()
        },
        "group_strategy": "original_"
        + "_".join(TEXT_COLS).lower()
        + "_tokenized_connected_components_v1",
        "fixed_training": {
            "optim": "adamw_torch",
            "warmup_ratio": 0.10,
            "early_stopping_patience": 1,
            "refit_epoch_strategy": "median_inner_best_epoch_rounded",
            "curvas": "train_eval_loss_epoch_csv_best_outer_png_v1",
            "checkpoint": "epoch_inner_and_refit_save_total_limit_2_v1",
        },
    }


def verificar_configuracion():
    ruta = RUN_DIR / "config_ejecucion.json"
    actual = configuracion_actual()
    if ruta.exists():
        anterior = json.loads(ruta.read_text(encoding="utf-8"))
        if anterior != actual:
            raise ValueError(
                "Esta carpeta tiene una corrida con datos/configuración diferentes. "
                "No mezcles resultados; cambia MODO/carpeta o archiva el experimento."
            )
    else:
        tmp = ruta.with_suffix(".tmp")
        tmp.write_text(json.dumps(actual, indent=2, ensure_ascii=False), encoding="utf-8")
        os.replace(tmp, ruta)


def ruta_estudio(variante, nombre):
    return RUN_DIR / f"distilbeto_{variante}_{nombre}.db"


def cargar_study(variante, nombre):
    ruta = ruta_estudio(variante, nombre)
    if not ruta.exists():
        return None
    return optuna.load_study(
        study_name=f"DistilBETO_{variante}_{nombre}",
        storage=f"sqlite:///{ruta.resolve().as_posix()}",
    )


def contar_completados():
    n = 0
    for variante in VARIANTES:
        ruta_csv = RUN_DIR / f"distilbeto_detalle_{variante}.csv"
        completados = set()
        if ruta_csv.exists():
            completados = set(pd.read_csv(ruta_csv)["outer_fold"].astype(int))
        for outer in range(1, N_OUTER + 1):
            if outer in completados:
                n += N_TRIALS * N_INNER + 1
            else:
                st = cargar_study(variante, f"outer{outer}")
                if st is not None:
                    n += (
                        min(
                            N_TRIALS,
                            sum(t.state == optuna.trial.TrialState.COMPLETE for t in st.trials),
                        )
                        * N_INNER
                    )
    return n

### 7.1 Checkpoints por época y curvas de pérdida

Las carpetas de cada entrenamiento se identifican por **variante, Outer Fold, configuración exacta e Inner Fold**, no por número de trial: si Optuna debe reencolar un trial interrumpido, se pueden recuperar sus Inner Folds sin perderlos. El estado se guarda al finalizar cada época; la validación de cada entrenamiento usa solo su Inner Fold, nunca el test oficial.

Los CSV contienen `epoch`, `train_loss`, `val_loss` y `val_f1_macro`. Se generan gráficos por Outer Fold del **mejor trial según Inner CV**, con las tres particiones internas; al acabar cada variante se pueden consultar en `graphs/<variante>/`. Las épocas que no alcanzan todos los folds por early stopping se indican con `n_folds` en el CSV agregado.

In [ ]:
# ==============================================================
# CHECKPOINTS, CURVAS Y REANUDACIÓN — ESQUEMA COMÚN A LOS TRES TRANSFORMERS
# ==============================================================
def config_id(cfg):
    """ID compacto y determinista para evitar rutas largas en Windows."""
    prep = {"PYSENTIMIENTO": "P"}[cfg["preprocessing"]]
    lr = int(round(float(cfg["learning_rate"]) * 1e5))  # 5e-5 -> 5
    bs = int(cfg["batch_size"])
    ep = int(cfg["num_train_epochs"])
    wd = int(round(float(cfg["weight_decay"]) * 100))  # .05 -> 5
    loss = {"standard": "s", "balanced": "b"}[cfg["loss_mode"]]
    drop = int(round(float(cfg["classifier_dropout"]) * 10))  # .2 -> 2, .3 -> 3
    return f"{prep}_l{lr}_b{bs}_e{ep}_w{wd}_{loss}_d{drop}"


def rutas_inner_fit(variante, etiqueta, cfg, inner_fold):
    """
    Estructura común a los tres Transformers:
      ckpt/mx/outer1/S_l5_b16_e5_w5_s_d3/inner1/
      state/mx_outer1_S_l5_b16_e5_w5_s_d3_inner1_complete.json
      curves/mx_outer1_S_l5_b16_e5_w5_s_d3_inner1.csv
    """
    ident = config_id(cfg)
    base = f"{variante}_{etiqueta}_{ident}_inner{inner_fold}"
    return {
        "config_id": ident,
        "state": FIT_STATE_DIR / f"{base}_complete.json",
        "curve": CURVE_DIR / f"{base}.csv",
        "ckpt": CHECKPOINT_DIR / variante / etiqueta / ident / f"inner{inner_fold}",
    }


def cargar_estado_inner_si_completo(rutas):
    if not rutas["state"].exists():
        return None
    return json.loads(rutas["state"].read_text(encoding="utf-8"))


def json_atomico(obj, ruta):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_name(ruta.name + ".tmp")
    temporal.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(temporal, ruta)


def extraer_curva(log_history):
    """Empareja train/validation por época, sin tocar Outer Test."""
    filas = {}
    for entry in log_history:
        if "epoch" not in entry:
            continue

        epoch = round(float(entry["epoch"]), 6)
        if epoch <= 0:
            continue

        fila = filas.setdefault(
            epoch,
            {"epoch": epoch, "train_loss": np.nan, "val_loss": np.nan, "val_f1_macro": np.nan},
        )

        if "loss" in entry:
            fila["train_loss"] = float(entry["loss"])
        if "eval_loss" in entry:
            fila["val_loss"] = float(entry["eval_loss"])
        if "eval_f1_macro" in entry:
            fila["val_f1_macro"] = float(entry["eval_f1_macro"])

    if not filas:
        return pd.DataFrame(columns=["epoch", "train_loss", "val_loss", "val_f1_macro"])

    return pd.DataFrame(list(filas.values())).sort_values("epoch").reset_index(drop=True)


def obtener_mejor_epoca_desde_curva(curva, max_epochs):
    if curva.empty or "val_f1_macro" not in curva or curva["val_f1_macro"].dropna().empty:
        return int(max_epochs)

    fila = curva.loc[curva["val_f1_macro"].idxmax()]
    return max(1, min(int(max_epochs), int(round(float(fila["epoch"])))))


def limpiar_checkpoint_dir(carpeta):
    if CLEAN_COMPLETED_CHECKPOINTS:
        shutil.rmtree(carpeta, ignore_errors=True)


def ejecutar_inner_fit_reanudable(
    cache, y, idx_train, idx_val, cfg, variante, etiqueta, inner_fold, seed_modelo, monitor=None
):
    rutas = rutas_inner_fit(variante, etiqueta, cfg, inner_fold)

    previo = cargar_estado_inner_si_completo(rutas)
    if previo is not None:
        if previo.get("config_id") != rutas["config_id"]:
            raise ValueError("Estado Inner almacenado no coincide con la configuración.")

        if monitor:
            monitor.fit_terminado(
                f"REUTILIZADO {variante.upper()} {etiqueta} "
                f"cfg={rutas['config_id']} inner={inner_fold}: "
                f"F1={float(previo['f1_inner']):.4f}"
            )

        return (float(previo["f1_inner"]), int(previo["best_epoch"]), rutas)

    rutas["ckpt"].mkdir(parents=True, exist_ok=True)

    dataset_completo = cache[cfg["preprocessing"]]
    train_ds = Subset(dataset_completo, list(map(int, idx_train)))
    val_ds = Subset(dataset_completo, list(map(int, idx_val)))

    if monitor:
        monitor.iniciar(
            f"{variante.upper()} {etiqueta} | "
            f"cfg={rutas['config_id']} | "
            f"Inner {inner_fold}/{N_INNER}"
        )

    model = crear_modelo(seed_modelo, cfg["classifier_dropout"])

    class_weights = pesos_balanceados(y[idx_train]) if cfg["loss_mode"] == "balanced" else None

    trainer = None
    try:
        args = argumentos_entrenamiento(rutas["ckpt"], cfg, seed_modelo, con_validacion=True)

        callbacks = [EarlyStoppingCallback(early_stopping_patience=1)]

        if monitor:
            callbacks.append(PulsoTrainer(monitor))

        trainer = entrenador_silencioso(
            ClasificadorTrainer(
                model=model,
                args=args,
                train_dataset=train_ds,
                eval_dataset=val_ds,
                data_collator=data_collator,
                compute_metrics=metricas,
                callbacks=callbacks,
                class_weights=class_weights,
            )
        )

        ultimo = get_last_checkpoint(str(rutas["ckpt"]))

        if ultimo and monitor:
            monitor.registrar(f"Reanudando fit desde {ultimo}")

        trainer.train(resume_from_checkpoint=(ultimo if ultimo else None))

        # Congelar la historia de entrenamiento ANTES de cualquier evaluación extra.
        curva = extraer_curva(trainer.state.log_history)

        if curva.empty or curva["val_loss"].isna().all() or curva["train_loss"].isna().all():
            raise RuntimeError("Faltan las curvas train/validation loss.")

        csv_atomico(curva, rutas["curve"])

        best_epoch = obtener_mejor_epoca_desde_curva(curva, cfg["num_train_epochs"])

        if trainer.state.best_metric is not None:
            f1 = float(trainer.state.best_metric)
        else:
            metrics = trainer.evaluate()
            f1 = float(metrics["eval_f1_macro"])

        json_atomico(
            {
                "status": "complete",
                "config_id": rutas["config_id"],
                "config": cfg,
                "f1_inner": f1,
                "best_epoch": int(best_epoch),
                "inner_fold": int(inner_fold),
                "seed": int(seed_modelo),
                "checkpoint_used_to_resume": ultimo,
            },
            rutas["state"],
        )

    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()

    # Al acabar correctamente, borrar únicamente los
    # checkpoints pesados. El JSON y la curva quedan fuera de esa carpeta.
    limpiar_checkpoint_dir(rutas["ckpt"])

    if monitor:
        monitor.fit_terminado(
            f"{variante.upper()} {etiqueta} "
            f"cfg={rutas['config_id']} "
            f"Inner {inner_fold}: F1={f1:.4f}"
        )

    return f1, best_epoch, rutas


def graficar_mejor_trial(variante, outer_fold, cfg, n_inner, mostrar=False):
    """Usa solo curvas de Inner Validation del mejor trial."""
    etiqueta = f"outer{outer_fold}"
    curves = []

    for fold in range(1, n_inner + 1):
        ruta = rutas_inner_fit(variante, etiqueta, cfg, fold)["curve"]

        if not ruta.exists():
            raise FileNotFoundError(f"Falta curva del mejor trial: {ruta}")

        d = pd.read_csv(ruta)
        d["inner_fold"] = fold
        curves.append(d)

    todos = pd.concat(curves, ignore_index=True)

    resumen = todos.groupby("epoch", as_index=False).agg(
        train_loss=("train_loss", "mean"),
        val_loss=("val_loss", "mean"),
        train_std=("train_loss", "std"),
        val_std=("val_loss", "std"),
        val_f1_macro=("val_f1_macro", "mean"),
        n_folds=("inner_fold", "nunique"),
    )

    salida = GRAPH_DIR / variante
    salida.mkdir(parents=True, exist_ok=True)

    csv_atomico(todos, salida / f"outer{outer_fold}_inner_folds.csv")
    csv_atomico(resumen, salida / f"outer{outer_fold}_media_por_epoca.csv")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(resumen["epoch"], resumen["train_loss"], marker="o", label="Train loss")
    ax.plot(resumen["epoch"], resumen["val_loss"], marker="o", label="Validation loss")
    ax.set(
        xlabel="Época",
        ylabel="Loss",
        title=(f"DistilBETO {variante.upper()} — Outer {outer_fold} | mejor trial interno"),
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    loss_png = salida / f"outer{outer_fold}_loss.png"
    fig.savefig(loss_png, dpi=160, bbox_inches="tight")
    if mostrar:
        display(fig)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(resumen["epoch"], resumen["val_f1_macro"], marker="o")
    ax.set(
        xlabel="Época",
        ylabel="F1-Macro validación",
        title=(f"DistilBETO {variante.upper()} — Outer {outer_fold} | mejor trial interno"),
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()

    f1_png = salida / f"outer{outer_fold}_f1.png"
    fig.savefig(f1_png, dpi=160, bbox_inches="tight")
    if mostrar:
        display(fig)
    plt.close(fig)

    return {
        "csv_inner": str(salida / f"outer{outer_fold}_inner_folds.csv"),
        "csv_media": str(salida / f"outer{outer_fold}_media_por_epoca.csv"),
        "loss_png": str(loss_png),
        "f1_png": str(f1_png),
    }

## 8. Búsqueda interna: TPE adaptativo con Optuna

Cada estudio (uno por Outer Fold) prueba 20 configuraciones: 5 propuestas iniciales aleatorias y hasta 15 adaptativas. Cada configuración se evalúa con **tres Inner Folds**, y la medida que se maximiza es su F1-Macro promedio. Como el refit usa la mediana de las mejores épocas internas, el Outer Test nunca determina ni las épocas ni los hiperparámetros.

Se mantiene la misma semilla de inicialización por Inner Fold entre trials para reducir ruido en la comparación (cada modelo parte del checkpoint preentrenado original).

**Espacio de búsqueda común a BETO, DistilBETO y RoBERTuito:** `learning_rate` ∈ {2e-5, 3e-5, 4e-5, 5e-5}, `batch_size` ∈ {16, 32}, `num_train_epochs` ∈ {3, 4, 5}, `weight_decay` ∈ {0, 0.01, 0.05, 0.1}, `loss_mode` ∈ {standard, balanced} y `classifier_dropout` (dropout de la cabeza de clasificación) ∈ {0.1, 0.2, 0.3}. Los tres reciben el mismo texto (`MESSAGE_TRANSFORMER`), por lo que el preprocesamiento no forma parte de la búsqueda.

**Recuperación:** SQLite guarda trials completos, los CSV muestran el historial y un archivo del sampler conserva su estado para continuaciones. Un trial interrumpido se vuelve a encolar con los mismos hiperparámetros: sus Inner Folds ya terminados se reutilizan y el que estaba en curso continúa desde su último checkpoint.

In [ ]:
def sugerir_config(trial):
    return {
        "preprocessing": trial.suggest_categorical("preprocessing", SEARCH_SPACE["preprocessing"]),
        "learning_rate": trial.suggest_categorical("learning_rate", SEARCH_SPACE["learning_rate"]),
        "batch_size": trial.suggest_categorical("batch_size", SEARCH_SPACE["batch_size"]),
        "num_train_epochs": trial.suggest_categorical(
            "num_train_epochs", SEARCH_SPACE["num_train_epochs"]
        ),
        "weight_decay": trial.suggest_categorical("weight_decay", SEARCH_SPACE["weight_decay"]),
        "loss_mode": trial.suggest_categorical("loss_mode", SEARCH_SPACE["loss_mode"]),
        "classifier_dropout": trial.suggest_categorical(
            "classifier_dropout", SEARCH_SPACE["classifier_dropout"]
        ),
    }


def buscar_en_train(
    cache, y, groups, indices, variante, nombre, k_inner, n_trials, seed_cv, monitor=None
):
    indices = np.asarray(indices, dtype=int)
    y_inner = y[indices]
    groups_inner = np.asarray(groups)[indices]

    skf = StratifiedGroupKFold(n_splits=k_inner, shuffle=True, random_state=seed_cv)

    splits = list(skf.split(np.zeros(len(indices)), y_inner, groups=groups_inner))

    for numero, (idx_tr_local, idx_val_local) in enumerate(splits, start=1):
        g_tr = set(groups_inner[idx_tr_local])
        g_val = set(groups_inner[idx_val_local])

        if g_tr & g_val:
            raise AssertionError(
                f"{variante} {nombre} "
                f"Inner {numero}: GROUP_ID "
                "compartido entre train y validación."
            )

    def objetivo(trial):
        cfg = sugerir_config(trial)
        scores = []
        epocas = []

        for numero, (idx_tr_local, idx_val_local) in enumerate(splits, start=1):
            tr = indices[idx_tr_local]
            val = indices[idx_val_local]

            # La misma semilla por Inner Fold entre configuraciones
            # reduce ruido al comparar hiperparámetros.
            seed_fold = SEED + seed_cv * 100 + numero

            f1, best_epoch, _ = ejecutar_inner_fit_reanudable(
                cache=cache,
                y=y,
                idx_train=tr,
                idx_val=val,
                cfg=cfg,
                variante=variante,
                etiqueta=nombre,
                inner_fold=numero,
                seed_modelo=seed_fold,
                monitor=monitor,
            )

            scores.append(float(f1))
            epocas.append(int(best_epoch))

        trial.set_user_attr("f1_inner_std", float(np.std(scores, ddof=1)))
        trial.set_user_attr("best_epochs_inner", [int(e) for e in epocas])
        trial.set_user_attr(
            "refit_epochs", int(np.clip(round(np.median(epocas)), 1, cfg["num_train_epochs"]))
        )

        return float(np.mean(scores))

    sampler_file = RUN_DIR / f"sampler_{variante}_{nombre}.pkl"

    if sampler_file.exists():
        with sampler_file.open("rb") as f:
            sampler = pickle.load(f)
    else:
        sampler = optuna.samplers.TPESampler(seed=seed_cv, n_startup_trials=N_STARTUP_TRIALS)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name=(f"DistilBETO_{variante}_{nombre}"),
        storage=(f"sqlite:///{ruta_estudio(variante, nombre).resolve().as_posix()}"),
        load_if_exists=True,
    )

    # Recuperar trials huérfanos tras una interrupción.
    running = [t for t in study.trials if t.state == optuna.trial.TrialState.RUNNING]

    waiting = [t for t in study.trials if t.state == optuna.trial.TrialState.WAITING]

    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

    for interrumpido in running:
        ya_reencolado = any(
            (
                t.state == optuna.trial.TrialState.WAITING
                and t.system_attrs.get("fixed_params", {}) == interrumpido.params
            )
            or (t.state == optuna.trial.TrialState.COMPLETE and t.params == interrumpido.params)
            for t in waiting + done
        )

        if not ya_reencolado and interrumpido.params:
            study.enqueue_trial(interrumpido.params)

        try:
            study.tell(
                interrumpido.number, state=(optuna.trial.TrialState.FAIL), skip_if_finished=True
            )
        except (ValueError, RuntimeError) as exc:
            raise RuntimeError(
                "No se pudo cerrar el trial " f"interrumpido {interrumpido.number}."
            ) from exc

        if monitor:
            monitor.registrar(f"Trial interrumpido {interrumpido.number} " "reencolado.")

    # Un trial FAIL con estado/curva/checkpoint parcial se reencola.
    for anterior in list(study.trials):
        if anterior.state != optuna.trial.TrialState.FAIL or not anterior.params:
            continue

        tiene_avance = False

        for numero in range(1, k_inner + 1):
            rutas = rutas_inner_fit(variante, nombre, anterior.params, numero)

            if rutas["state"].exists() or get_last_checkpoint(str(rutas["ckpt"])) is not None:
                tiene_avance = True
                break

        if not tiene_avance:
            continue

        ya_pendiente_o_completo = any(
            (t.state == optuna.trial.TrialState.COMPLETE and t.params == anterior.params)
            or (
                t.state == optuna.trial.TrialState.WAITING
                and t.system_attrs.get("fixed_params", {}) == anterior.params
            )
            for t in study.trials
        )

        if not ya_pendiente_o_completo:
            study.enqueue_trial(anterior.params)

            if monitor:
                monitor.registrar(f"Reencolado trial FAIL {anterior.number} con " "avance parcial.")

    completos = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)

    faltantes = max(0, n_trials - completos)

    if monitor and completos:
        monitor.registrar(
            f"Recuperados {completos} trials de {variante} {nombre}; faltan {faltantes}."
        )

    def trial_finalizado(st, trial):
        tmp_sampler = sampler_file.with_suffix(".pkl.tmp")

        with tmp_sampler.open("wb") as f:
            pickle.dump(st.sampler, f)

        os.replace(tmp_sampler, sampler_file)

        csv_atomico(st.trials_dataframe(), RUN_DIR / f"trials_{variante}_{nombre}.csv")

        n_completos = sum(t.state == optuna.trial.TrialState.COMPLETE for t in st.trials)

        if trial.state == optuna.trial.TrialState.COMPLETE:
            msg = (
                f"{variante.upper()} {nombre} | "
                f"trial {n_completos}/{n_trials} "
                f"F1_inner={trial.value:.4f} | "
                f"mejor={st.best_value:.4f}"
            )
        else:
            msg = f"{variante.upper()} {nombre} | trial ID {trial.number} {trial.state.name}"

        if monitor:
            monitor.registrar(msg)
            monitor.mostrar(msg, forzar=True)

    if faltantes:
        study.optimize(
            objetivo,
            n_trials=faltantes,
            callbacks=[trial_finalizado],
            show_progress_bar=False,
            gc_after_trial=True,
        )

    completos_final = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)

    if completos_final < n_trials:
        raise RuntimeError("No se completaron todos los trials; consulta monitor.log.")

    return study

## 9. Refit externo sin utilizar Outer Test para early stopping

Se crea un DistilBETO nuevo y se entrena sobre todo `Outer Train`, durante la mediana de las mejores épocas internas. El `Outer Test` se usa solo para calcular F1 y las demás métricas. Se registran F1 train, F1 inner, F1 outer y gap.

In [ ]:
def refit_externo(
    cache, y, tr, out, cfg, epocas, seed_modelo, monitor=None, variante=None, outer_fold=None
):
    if variante is None or outer_fold is None:
        raise ValueError("Refit externo necesita variante y outer_fold.")

    state_path = FIT_STATE_DIR / f"{variante}_outer{outer_fold}_refit_complete.json"

    curve_path = CURVE_DIR / f"{variante}_outer{outer_fold}_refit.csv"

    ckpt_dir = CHECKPOINT_DIR / variante / f"outer{outer_fold}" / "refit"

    cfg_id = config_id(cfg)

    if state_path.exists():
        previo = json.loads(state_path.read_text(encoding="utf-8"))

        if previo.get("config_id") != cfg_id:
            raise ValueError("Refit almacenado no coincide con la configuración seleccionada.")

        if monitor:
            monitor.fit_terminado(f"REUTILIZADO refit {variante.upper()} outer {outer_fold}")

        return {
            k: float(previo[k])
            for k in ("f1_train", "f1_outer_clase_0", "f1_outer_clase_1", "f1_outer", "gap")
        }

    ds = cache[cfg["preprocessing"]]

    ckpt_dir.mkdir(parents=True, exist_ok=True)

    if monitor:
        monitor.iniciar(
            f"{variante.upper()} "
            f"Outer {outer_fold}: "
            f"refit {cfg['preprocessing']}, "
            f"{epocas} épocas"
        )

    model = crear_modelo(seed_modelo, cfg["classifier_dropout"])

    cw = pesos_balanceados(y[tr]) if cfg["loss_mode"] == "balanced" else None

    trainer = None
    try:
        args = argumentos_entrenamiento(
            ckpt_dir, cfg, seed_modelo, con_validacion=False, epochs=epocas
        )

        callbacks = [PulsoTrainer(monitor)] if monitor else []

        trainer = entrenador_silencioso(
            ClasificadorTrainer(
                model=model,
                args=args,
                train_dataset=Subset(ds, tr.tolist()),
                data_collator=data_collator,
                class_weights=cw,
                callbacks=callbacks,
            )
        )

        ultimo = get_last_checkpoint(str(ckpt_dir))

        if ultimo and monitor:
            monitor.registrar(f"Reanudando Outer refit desde {ultimo}")

        trainer.train(resume_from_checkpoint=(ultimo if ultimo else None))

        curva = extraer_curva(trainer.state.log_history)

        if not curva.empty:
            csv_atomico(curva, curve_path)

        p_train = predecir(trainer, Subset(ds, tr.tolist()))

        p_outer = predecir(trainer, Subset(ds, out.tolist()))

        f_train = float(f1_score(y[tr], p_train, average="macro"))

        f_outer_clases = f1_score(y[out], p_outer, labels=[0, 1], average=None, zero_division=0)

        f_outer = float(f1_score(y[out], p_outer, average="macro"))

        resultado = {
            "f1_train": f_train,
            "f1_outer_clase_0": float(f_outer_clases[0]),
            "f1_outer_clase_1": float(f_outer_clases[1]),
            "f1_outer": f_outer,
            "gap": f_train - f_outer,
        }

        json_atomico({"status": "complete", "config_id": cfg_id, **resultado}, state_path)

    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()

    limpiar_checkpoint_dir(ckpt_dir)

    if monitor:
        monitor.fit_terminado(
            f"{variante.upper()} "
            f"Outer {outer_fold} refit: "
            f"F1_train={f_train:.4f}, "
            f"F1_outer={f_outer:.4f}"
        )

    return resultado

## 10. Nested CV por variante (reanudar sin perder Outer Folds)

In [ ]:
def ejecutar_variante(variante, monitor):
    df = TRAIN_AGRUPADO[variante].copy()
    cache, y = cache_dataset(df)
    groups = df[GROUP_COL].astype(str).to_numpy()
    outer_cv = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)
    splits = list(outer_cv.split(np.zeros(len(y)), y, groups=groups))
    for numero, (tr_chk, out_chk) in enumerate(splits, start=1):
        if set(groups[tr_chk]) & set(groups[out_chk]):
            raise AssertionError(f"{variante} Outer {numero}: GROUP_ID compartido train/test.")
    ruta_parcial = RUN_DIR / f"distilbeto_detalle_{variante}.csv"
    if ruta_parcial.exists():
        resultados = pd.read_csv(ruta_parcial).to_dict("records")
        folds_hechos = {int(x["outer_fold"]) for x in resultados}
    else:
        resultados, folds_hechos = [], set()

    for outer_fold, (tr, out) in enumerate(splits, start=1):
        if outer_fold in folds_hechos:
            msg = f"[{variante.upper()}][DistilBETO][Outer {outer_fold}/{N_OUTER}] resultado recuperado; se omite."
            print(msg)
            monitor.registrar(msg)
            continue
        monitor.iniciar(f"{variante.upper()} | Outer {outer_fold}/{N_OUTER}")
        study = buscar_en_train(
            cache,
            y,
            groups,
            tr,
            variante,
            f"outer{outer_fold}",
            N_INNER,
            N_TRIALS,
            seed_cv=SEED + outer_fold,
            monitor=monitor,
        )
        cfg = study.best_params.copy()
        refit_epochs = int(study.best_trial.user_attrs["refit_epochs"])
        ext = refit_externo(
            cache,
            y,
            tr,
            out,
            cfg,
            refit_epochs,
            seed_modelo=SEED + 1000 + outer_fold,
            monitor=monitor,
            variante=variante,
            outer_fold=outer_fold,
        )
        # Graficar solamente las curvas del MEJOR TRIAL INTERNO del Outer Fold.
        graficas = graficar_mejor_trial(variante, outer_fold, cfg, N_INNER, mostrar=False)
        fila = {
            "variante": variante,
            "outer_fold": outer_fold,
            "best_preprocessing": cfg["preprocessing"],
            "f1_inner": float(study.best_value),
            "std_inner": float(study.best_trial.user_attrs["f1_inner_std"]),
            "refit_epochs": refit_epochs,
            "f1_train": ext["f1_train"],
            "f1_outer_clase_0": ext["f1_outer_clase_0"],
            "f1_outer_clase_1": ext["f1_outer_clase_1"],
            "f1_outer": ext["f1_outer"],
            "gap": ext["gap"],
            "grafica_loss": graficas["loss_png"],
            "grafica_f1": graficas["f1_png"],
            "best_params": json.dumps(cfg, ensure_ascii=False),
        }
        resultados.append(fila)
        csv_atomico(pd.DataFrame(resultados).sort_values("outer_fold"), ruta_parcial)
        msg = (
            f"[{variante.upper()}][DistilBETO][Outer {outer_fold}/{N_OUTER}] FINALIZADO | "
            f"prep={fila['best_preprocessing']} | "
            f"F1_inner={fila['f1_inner']:.4f}±{fila['std_inner']:.4f} | "
            f"F1_train={fila['f1_train']:.4f} | "
            f"F1_outer={fila['f1_outer']:.4f} | gap={fila['gap']:.4f}"
        )
        print(msg, flush=True)
        monitor.registrar(msg)
        monitor.mostrar(msg, forzar=True)

    detalle = pd.DataFrame(resultados).sort_values("outer_fold").reset_index(drop=True)
    resumen = {
        "variante": variante,
        "n": len(y),
        "n_groups": int(df[GROUP_COL].nunique()),
        "f1_train_mean": float(detalle["f1_train"].mean()),
        "f1_nested_mean": float(detalle["f1_outer"].mean()),
        "f1_nested_std": float(detalle["f1_outer"].std(ddof=1)),
        "f1_nested_min": float(detalle["f1_outer"].min()),
        "f1_nested_max": float(detalle["f1_outer"].max()),
        "gap_mean": float(detalle["gap"].mean()),
    }
    return detalle, resumen

## 11. Ejecutar TPE20 con seguimiento y checkpoints

Tras `MODO="prueba"`, cambia a `"completo"`, reinicia el kernel y ejecuta desde el inicio. La corrida completa mantiene 5 Outer × 3 Inner × 20 trials. La salida visual sigue compacta; los CSV de pérdidas se guardan al completar cada Inner Fold. Los gráficos PNG del mejor trial se generan al finalizar cada Outer Fold.

**Para reanudar:** usa exactamente **este notebook**, mismo `MODO` y mismos datos. SQLite reencola los trials interrumpidos, los Inner Folds ya terminados no se repiten y `Trainer` reanuda desde el checkpoint más reciente de la época anterior. No borres la carpeta de la corrida (`seguimiento/<RUN_TAG>/<modo>/`) mientras el experimento esté incompleto. Los checkpoints pesados se eliminan solo después de confirmar que se guardaron resultados y curvas.

In [ ]:
if not EJECUTAR_NESTED:
    print(
        "[DistilBETO] Nested CV desactivado. Ejecuta primero el pre-flight; luego cambia EJECUTAR_NESTED=True."
    )
    resultados_detalle = {}
    resultados_resumen = []
    resumen_nested = pd.DataFrame()
else:
    verificar_configuracion()
    total_fits = len(VARIANTES) * N_OUTER * (N_INNER * N_TRIALS + 1)
    monitor = Monitor(RUN_DIR, total=total_fits, ya_completados=contar_completados())
    resultados_detalle, resultados_resumen = {}, []
    try:
        for variante in VARIANTES:
            detalle, resumen = ejecutar_variante(variante, monitor)
            resultados_detalle[variante] = detalle
            resultados_resumen.append(resumen)
            csv_atomico(
                pd.DataFrame(resultados_resumen), RUN_DIR / "distilbeto_resumen_parcial.csv"
            )
    except KeyboardInterrupt:
        monitor.fallar("Nested CV interrumpido; los trials completos quedan en SQLite")
        raise
    except Exception as exc:
        monitor.registrar(f"ERROR {type(exc).__name__}: {exc}")
        monitor.fallar(f"Nested CV con error {type(exc).__name__}; consultar monitor.log")
        raise
    else:
        monitor.finalizar(f"Nested CV completado ({', '.join(v.upper() for v in VARIANTES)})")

    resumen_nested = pd.DataFrame(resultados_resumen)
    print("\n[DistilBETO] RESUMEN NESTED CV")
    display(resumen_nested)
    for variante, detalle in resultados_detalle.items():
        print(f"\nDetalle {variante.upper()}")
        display(detalle.drop(columns=["best_params"]))

## 12. Guardar resultados de la búsqueda

Durante la corrida ya se guardaron los CSV por Outer Fold y las bases SQLite por estudio. Esta celda consolida el resumen y el detalle de las tres variantes al finalizar, sin mirar el test oficial.

In [ ]:
if not EJECUTAR_NESTED:
    print("No se guardan resultados Nested porque EJECUTAR_NESTED=False.")
elif MODO == "completo":
    csv_atomico(resumen_nested, RESULTS_DIR / "distilbeto_nestedcv_resumen.csv")
    for variante in VARIANTES:
        csv_atomico(
            resultados_detalle[variante],
            RESULTS_DIR / f"distilbeto_nestedcv_detalle_{variante}.csv",
        )
    print("Resultados completos disponibles en:", RESULTS_DIR)
else:
    print("Solo resultados de prueba, aislados en:", RUN_DIR)

## 13. Diagnóstico posterior de Optuna (sin entrenar)

Una vez terminada la corrida completa, esta celda recopila el historial de los 5 estudios Outer × 3 variantes = **15 estudios**, con hasta 20 trials completos cada uno (**300 trials** en total), e identifica en qué número de trial apareció la mejor F1 interna. Para el mejor trial de cada estudio muestra también sus hiperparámetros. **Son diagnósticos descriptivos, no una segunda selección basada en Outer Test.**

In [ ]:
# Solo análisis de los resultados que YA están en CSV; no toca la GPU.
if MODO != "completo":
    print("Ejecuta este diagnóstico una vez acabada la corrida COMPLETA.")
else:
    historiales = []
    for variante in VARIANTES:
        for fold in range(1, N_OUTER + 1):
            ruta = RUN_DIR / f"trials_{variante}_outer{fold}.csv"
            if ruta.exists():
                temp = pd.read_csv(ruta)
                temp["variante"] = variante
                temp["outer_fold"] = fold
                historiales.append(temp)
    if historiales:
        df_historial = pd.concat(historiales, ignore_index=True)
        csv_atomico(df_historial, RESULTS_DIR / "distilbeto_optuna_historial_consolidado.csv")
        completados = df_historial[df_historial["state"] == "COMPLETE"].copy()
        print(
            f"Estudios detectados: {len(historiales)}/15; trials completos: {len(completados)}/300"
        )
        # Best trial interno por estudio: no comparar esta puntuación entre datos diferentes.
        mejores = (
            completados.sort_values("value", ascending=False)
            .drop_duplicates(["variante", "outer_fold"])
            .sort_values(["variante", "outer_fold"])
        )
        columnas = ["variante", "outer_fold", "number", "value"] + [
            col for col in completados if col.startswith("params_")
        ]
        display(mejores[columnas].reset_index(drop=True))
        csv_atomico(mejores[columnas], RESULTS_DIR / "distilbeto_optuna_mejores_trials.csv")
    else:
        print("Todavía no hay historiales. Completa el Nested CV primero.")

## 14. Curvas por Outer Fold (sin volver a entrenar)

In [ ]:
# Modifica estos dos valores para revisar las curvas del mejor trial de un fold.
VARIANTE_GRAFICA = "mx"
OUTER_GRAFICA = 1

from IPython.display import Image

ruta_loss = GRAPH_DIR / VARIANTE_GRAFICA / f"outer{OUTER_GRAFICA}_loss.png"
ruta_f1 = GRAPH_DIR / VARIANTE_GRAFICA / f"outer{OUTER_GRAFICA}_f1.png"
ruta_csv = GRAPH_DIR / VARIANTE_GRAFICA / f"outer{OUTER_GRAFICA}_media_por_epoca.csv"

if ruta_loss.exists() and ruta_f1.exists():
    print(f"[{VARIANTE_GRAFICA.upper()}][DistilBETO][Outer {OUTER_GRAFICA}] Curvas guardadas")
    display(Image(filename=str(ruta_loss)))
    display(Image(filename=str(ruta_f1)))
    display(pd.read_csv(ruta_csv))
else:
    print("Aún no están disponibles ambas curvas para ese Outer Fold.")

## 15. Optimización final (solo train)

Se ejecuta automáticamente después de la Nested CV (`EJECUTAR_OPTIMIZACION_FINAL=True`) y solo en modo completo. Esta sección lanza 20 trials × 3 Inner Folds × 3 variantes = **180 entrenamientos adicionales**, selecciona una configuración por variante usando **solo train** y la guarda en JSON. Se hace antes de abrir el test oficial; si se interrumpe, al volver a ejecutar se reanuda desde los trials guardados.

In [ ]:
def contar_completados_final():
    """Fine-tunings de la optimización final ya terminados (trials completos × 3 Inner)."""
    n = 0
    for variante in VARIANTES:
        st = cargar_study(variante, "final")
        if st is not None:
            n += (
                min(N_TRIALS, sum(t.state == optuna.trial.TrialState.COMPLETE for t in st.trials))
                * 3
            )
    return n


configs_finales = {}
if EJECUTAR_OPTIMIZACION_FINAL and MODO != "completo":
    print("Optimización final omitida en MODO='prueba' (solo se ejecuta en modo completo).")
elif EJECUTAR_OPTIMIZACION_FINAL:
    verificar_configuracion()
    # Mismo panel de seguimiento que en Nested CV: 20 trials × 3 Inner por variante.
    monitor_final = Monitor(
        RUN_DIR, total=len(VARIANTES) * N_TRIALS * 3, ya_completados=contar_completados_final()
    )
    try:
        for variante in VARIANTES:
            monitor_final.iniciar(f"{variante.upper()} | Optimización final")
            df = TRAIN_AGRUPADO[variante].copy()
            cache, y = cache_dataset(df)
            groups = df[GROUP_COL].astype(str).to_numpy()
            estudio = buscar_en_train(
                cache,
                y,
                groups,
                np.arange(len(y)),
                variante,
                "final",
                k_inner=3,
                n_trials=N_TRIALS,
                seed_cv=SEED,
                monitor=monitor_final,
            )
            cfg = estudio.best_params.copy()
            cfg["refit_epochs"] = int(estudio.best_trial.user_attrs["refit_epochs"])
            configs_finales[variante] = cfg
            print(f"{variante.upper()}: F1-CV final={estudio.best_value:.4f}; config={cfg}")
    except KeyboardInterrupt:
        monitor_final.fallar(
            "Optimización final interrumpida; los trials completos quedan en SQLite"
        )
        raise
    except Exception as exc:
        monitor_final.registrar(f"ERROR optimización final {type(exc).__name__}: {exc}")
        monitor_final.fallar(
            f"Optimización final con error {type(exc).__name__}; consultar monitor.log"
        )
        raise
    monitor_final.finalizar("Optimización final completada")
    (RESULTS_DIR / "distilbeto_config_final.json").write_text(
        json.dumps(configs_finales, indent=2, ensure_ascii=False), encoding="utf-8"
    )
else:
    print("Optimización final desactivada. Analiza primero los Outer Folds.")

## 16. Curvas de entrenamiento de la configuración final (figura del informe)

Una figura por modelo con un panel por variante (MX, ES, CU), usando la **configuración final** de cada variante:

- fila superior: pérdida de train y de validación por época;
- fila inferior: F1-Macro de validación por época;
- línea y banda: media ± desviación estándar de los 3 Inner Folds;
- marcador hueco: época que solo alcanzó un Inner Fold por el early stopping;
- línea punteada: número de épocas usado en el refit.

Se guarda en PNG (para el documento) y PDF (vectorial), junto con un CSV con los valores. No entrena nada.

In [ ]:
# ============================================================
# CURVAS DE ENTRENAMIENTO — CONFIGURACIÓN FINAL (FIGURA DEL INFORME)
# Una figura por modelo y un panel por variante:
#   fila 1 = pérdida de train y de validación;
#   fila 2 = F1-Macro de validación.
# Media ± desviación estándar de los 3 Inner Folds de la optimización final.
# Solo lee CSV ya guardados: no entrena nada.
# ============================================================
NOMBRE_MODELO = "DistilBETO"

ruta_cfg = RESULTS_DIR / "distilbeto_config_final.json"
if not configs_finales and ruta_cfg.exists():
    configs_finales = json.loads(ruta_cfg.read_text(encoding="utf-8"))
if not configs_finales:
    print("Aún no hay configuración final (se genera en la sección 15, en modo completo).")
else:
    salida_final = GRAPH_DIR / "config_final"
    salida_final.mkdir(parents=True, exist_ok=True)

    def curva_media_config_final(variante, cfg):
        """Promedia por época las curvas de los 3 Inner Folds de la configuración final."""
        cfg_busqueda = {k: v for k, v in cfg.items() if k != "refit_epochs"}
        partes = []
        for inner_fold in range(1, 4):
            ruta = rutas_inner_fit(variante, "final", cfg_busqueda, inner_fold)["curve"]
            if not ruta.exists():
                raise FileNotFoundError(f"Falta la curva del Inner Fold {inner_fold}: {ruta}")
            d = pd.read_csv(ruta)
            d["inner_fold"] = inner_fold
            partes.append(d)
        todas = pd.concat(partes, ignore_index=True)
        return todas.groupby("epoch", as_index=False).agg(
            train_loss=("train_loss", "mean"),
            train_std=("train_loss", "std"),
            val_loss=("val_loss", "mean"),
            val_std=("val_loss", "std"),
            val_f1_macro=("val_f1_macro", "mean"),
            f1_std=("val_f1_macro", "std"),
            n_folds=("inner_fold", "nunique"),
        )

    def dibujar_serie(ax, datos, col, col_std, etiqueta, estilo):
        """Línea + banda para épocas con ≥2 Inner Folds; marcador hueco si solo queda 1."""
        respaldado = datos[datos["n_folds"] >= 2]
        aislado = datos[datos["n_folds"] < 2]
        (linea,) = ax.plot(respaldado["epoch"], respaldado[col], estilo, marker="o", label=etiqueta)
        std = respaldado[col_std].fillna(0)
        ax.fill_between(
            respaldado["epoch"],
            respaldado[col] - std,
            respaldado[col] + std,
            alpha=0.15,
            color=linea.get_color(),
        )
        if not aislado.empty:
            ax.plot(
                aislado["epoch"],
                aislado[col],
                linestyle="none",
                marker="o",
                mfc="none",
                color=linea.get_color(),
            )

    variantes_fig = [v for v in ["mx", "es", "cu"] if v in configs_finales]
    fig, axes = plt.subplots(
        2,
        len(variantes_fig),
        figsize=(4.6 * len(variantes_fig), 7.4),
        sharex="col",
        sharey="row",
        squeeze=False,
    )
    tablas = []
    for j, variante in enumerate(variantes_fig):
        cfg = configs_finales[variante]
        resumen = curva_media_config_final(variante, cfg)
        resumen.insert(0, "variante", variante)
        tablas.append(resumen)
        refit = int(cfg["refit_epochs"])

        ax_loss, ax_f1 = axes[0, j], axes[1, j]
        dibujar_serie(ax_loss, resumen, "train_loss", "train_std", "Train", "-")
        dibujar_serie(ax_loss, resumen, "val_loss", "val_std", "Validación", "--")
        dibujar_serie(ax_f1, resumen, "val_f1_macro", "f1_std", "F1-Macro validación", "-")
        for ax in (ax_loss, ax_f1):
            ax.axvline(refit, color="gray", linestyle=":", linewidth=1.2, label="Épocas del refit")
            ax.grid(alpha=0.25)
            ax.set_xticks(range(1, int(resumen["epoch"].max()) + 1))
        ax_loss.set_title(f"{variante.upper()} · refit {refit} épocas")
        ax_f1.set_xlabel("Época")
        if j == 0:
            ax_loss.set_ylabel("Pérdida")
            ax_f1.set_ylabel("F1-Macro")
            ax_loss.legend(fontsize=8)
            ax_f1.legend(fontsize=8)

    fig.suptitle(
        f"{NOMBRE_MODELO} — curvas de entrenamiento de la configuración final\n"
        "(media ± d.e. de 3 Inner Folds; marcador hueco = época alcanzada por un solo fold)",
        fontsize=11,
    )
    fig.tight_layout()
    for carpeta in (salida_final, RESULTS_DIR):
        fig.savefig(carpeta / "distilbeto_curvas_entrenamiento.png", dpi=200, bbox_inches="tight")
        fig.savefig(carpeta / "distilbeto_curvas_entrenamiento.pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)

    df_curvas_finales = pd.concat(tablas, ignore_index=True)
    csv_atomico(df_curvas_finales, RESULTS_DIR / "distilbeto_curvas_entrenamiento.csv")
    for variante in variantes_fig:
        print(f"{variante.upper()} — configuración final: {configs_finales[variante]}")
    display(df_curvas_finales.round(4))

In [ ]:
from IPython.display import Image

ruta_figura = RESULTS_DIR / "distilbeto_curvas_entrenamiento.png"
if ruta_figura.exists():
    display(Image(filename=str(ruta_figura)))
else:
    print("Aún no se ha generado la figura: ejecuta la celda anterior tras la optimización final.")

## 17. Test oficial — bloqueado

No utilices los 600 tweets oficiales por variante hasta haber congelado el protocolo. Esta sección entrena 5 modelos nuevos por variante con semillas predeterminadas y reporta media ± desviación, sin seleccionar la mejor semilla según test. Se habilita únicamente con `EJECUTAR_TEST_OFICIAL=True` y `CONFIRMACION_TEST="TEST_OFICIAL_CONGELADO"`.

**Análisis de sensibilidad.** El preprocesamiento no retira del train los textos que coinciden con el test. Antes de evaluar, se marcan los textos del test que coinciden con alguno del train (mismo `MESSAGE` original o misma secuencia de tokens de entrada de DistilBETO) y se guardan en `audit/coincidencias_test_train_<variante>_<prep>.csv`. Para cada semilla se reporta el F1-Macro con los 600 textos y, con el mismo modelo, el F1-Macro sin esos textos. Este cálculo no interviene en ninguna decisión del modelo.

Además se guardan, por semilla y por texto, la predicción y la probabilidad de ironía en `distilbeto_test_probabilidades.csv`: son la base de la curva ROC, la curva Precision-Recall y las matrices de confusión del notebook de comparación final.

In [ ]:
def cargar_test(variante):
    ruta = DATA_DIR / f"test_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(ruta)
    df = pd.read_csv(ruta)
    req = [ID_COL, ORIGINAL_COL, LABEL_COL, *TEXT_COLS.values()]
    if any(col not in df.columns for col in req):
        raise ValueError(f"Test {variante} no contiene todas las columnas esperadas.")
    df[ID_COL] = df[ID_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    df[ORIGINAL_COL] = df[ORIGINAL_COL].fillna("").astype(str)
    for col in TEXT_COLS.values():
        df[col] = df[col].fillna("").astype(str)
    if len(df) != 600:
        raise ValueError(f"Se esperan 600 ejemplos del test oficial {variante}, hay {len(df)}.")
    return df.reset_index(drop=True)


def detectar_coincidencias_test(train_df, test_df, prep, variante):
    """Marca los textos del test que coinciden con alguno del train.

    Solo alimenta el análisis de sensibilidad del test oficial: no elimina
    registros ni interviene en ninguna decisión del modelo.
    Coincidencia = mismo MESSAGE original (espacios colapsados) o misma
    secuencia de tokens de entrada del modelo (truncada a MAX_LENGTH).
    """
    col = TEXT_COLS[prep]

    def claves_tokens(textos):
        ids = tokenizer(
            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False
        )["input_ids"]
        return [tuple(x) for x in ids]

    originales_train = set(train_df[ORIGINAL_COL].map(colapsar_espacios))
    tokens_train = set(claves_tokens(train_df[col]))
    coincide_original = (
        test_df[ORIGINAL_COL].map(colapsar_espacios).isin(originales_train).to_numpy()
    )
    coincide_tokens = np.array([k in tokens_train for k in claves_tokens(test_df[col])], dtype=bool)
    mascara = coincide_original | coincide_tokens

    detalle = test_df.loc[mascara, [ID_COL, ORIGINAL_COL, col]].copy()
    detalle["coincide_original"] = coincide_original[mascara]
    detalle["coincide_tokens"] = coincide_tokens[mascara]
    csv_atomico(detalle, AUDIT_DIR / f"coincidencias_test_train_{variante}_{prep}.csv")
    return mascara


def predecir_probabilidades(trainer, dataset):
    """Probabilidades softmax por clase (columna 1 = irónico)."""
    logits = trainer.predict(dataset).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    return probs / probs.sum(axis=1, keepdims=True)


def evaluar_semilla_final(train_df, test_df, cfg, seed, variante, mascara_coincide, monitor=None):
    prep = cfg["preprocessing"]
    ds_train = DatasetTokenizado(train_df[TEXT_COLS[prep]], train_df[LABEL_COL])
    ds_test = DatasetTokenizado(test_df[TEXT_COLS[prep]], test_df[LABEL_COL])
    y_train = train_df[LABEL_COL].to_numpy(dtype=int)
    y_test = test_df[LABEL_COL].to_numpy(dtype=int)
    sin_coincidencias = ~np.asarray(mascara_coincide, dtype=bool)
    carpeta = CHECKPOINT_DIR / variante / "test_final" / f"seed{seed}"
    carpeta.mkdir(parents=True, exist_ok=True)
    resultado_file = FIT_STATE_DIR / f"{variante}_test_seed{seed}_complete.json"
    cfg_id = config_id(cfg)
    claves = (
        "seed",
        "f1_macro",
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro_sin_coincidencias",
        "n_coincidencias",
    )
    pred_file = FIT_STATE_DIR / f"{variante}_test_seed{seed}_predicciones.csv"
    if resultado_file.exists():
        if not pred_file.exists():
            raise FileNotFoundError(
                f"Falta {pred_file}; borra {resultado_file} y repite esa semilla."
            )
        previo = json.loads(resultado_file.read_text(encoding="utf-8"))
        if previo.get("config_id") != cfg_id:
            raise ValueError("Configuración final distinta a la guardada para la semilla")
        return {k: previo.get(k) for k in claves}
    if monitor:
        monitor.iniciar(f"{variante.upper()} | Test oficial | semilla {seed}")
    cw = pesos_balanceados(y_train) if cfg["loss_mode"] == "balanced" else None
    model = crear_modelo(seed, cfg["classifier_dropout"])
    trainer = None
    try:
        args = argumentos_entrenamiento(
            carpeta, cfg, seed, con_validacion=False, epochs=cfg["refit_epochs"]
        )
        trainer = entrenador_silencioso(
            ClasificadorTrainer(
                model=model,
                args=args,
                train_dataset=ds_train,
                data_collator=data_collator,
                class_weights=cw,
                callbacks=[PulsoTrainer(monitor)] if monitor else [],
            )
        )
        ultimo = get_last_checkpoint(str(carpeta))
        trainer.train(resume_from_checkpoint=ultimo if ultimo else None)
        probs = predecir_probabilidades(trainer, ds_test)
        y_pred = probs.argmax(axis=1)
        # Predicciones y probabilidades por texto: base de la ROC, la curva PR
        # y las matrices de confusión del notebook de comparación final.
        csv_atomico(
            pd.DataFrame(
                {
                    "variante": variante,
                    "seed": int(seed),
                    "ID": test_df[ID_COL].astype(str).to_numpy(),
                    "y_true": y_test,
                    "y_pred": y_pred,
                    "prob_ironia": probs[:, 1],
                    "coincide_train": ~sin_coincidencias,
                }
            ),
            pred_file,
        )
        curva = extraer_curva(trainer.state.log_history)
        if not curva.empty:
            csv_atomico(curva, CURVE_DIR / f"{variante}_test_seed{seed}.csv")
        resultado = {
            "seed": int(seed),
            "f1_macro": float(f1_score(y_test, y_pred, average="macro")),
            "accuracy": float(accuracy_score(y_test, y_pred)),
            "precision_macro": float(
                precision_score(y_test, y_pred, average="macro", zero_division=0)
            ),
            "recall_macro": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
            # Análisis de sensibilidad: mismo modelo, sin los textos que coinciden con el train.
            "f1_macro_sin_coincidencias": float(
                f1_score(y_test[sin_coincidencias], y_pred[sin_coincidencias], average="macro")
            ),
            "n_coincidencias": int((~sin_coincidencias).sum()),
        }
        json_atomico({"config_id": cfg_id, **resultado}, resultado_file)
        limpiar_checkpoint_dir(carpeta)
    finally:
        if trainer is not None:
            del trainer
        del model
        liberar_memoria()
    if monitor:
        monitor.fit_terminado(
            f"{variante.upper()} test semilla {seed}: F1={resultado['f1_macro']:.4f}"
        )
    return resultado

In [ ]:
if EJECUTAR_TEST_OFICIAL:
    if MODO != "completo" or not EJECUTAR_OPTIMIZACION_FINAL:
        raise ValueError("Primero termina Nested CV y la optimización final en modo completo.")
    if CONFIRMACION_TEST != "TEST_OFICIAL_CONGELADO":
        raise ValueError("Bloqueado: confirma explícitamente la evaluación final.")
    datos_test = []
    monitor_test = Monitor(
        RUN_DIR,
        total=len(VARIANTES) * len(SEMILLAS_FINALES),
        ya_completados=sum(
            (FIT_STATE_DIR / f"{v}_test_seed{s}_complete.json").exists()
            for v in VARIANTES
            for s in SEMILLAS_FINALES
        ),
    )
    try:
        for variante in VARIANTES:
            df_train = cargar_train(variante)
            df_test = cargar_test(variante)
            cfg = configs_finales[variante]
            mascara = detectar_coincidencias_test(df_train, df_test, cfg["preprocessing"], variante)
            print(
                f"[TEST][DistilBETO][{variante.upper()}] textos del test que coinciden "
                f"con el train: {int(mascara.sum())}/{len(mascara)}"
            )
            for seed in SEMILLAS_FINALES:
                res = evaluar_semilla_final(
                    df_train, df_test, cfg, seed, variante, mascara, monitor=monitor_test
                )
                res["variante"] = variante
                datos_test.append(res)
                print(
                    f"{variante.upper()} seed {seed}: F1={res['f1_macro']:.4f} | "
                    f"F1 sin coincidencias={res['f1_macro_sin_coincidencias']:.4f}"
                )
                csv_atomico(
                    pd.DataFrame(datos_test), RESULTS_DIR / "distilbeto_test_5_semillas.csv"
                )
    except KeyboardInterrupt:
        monitor_test.fallar("Test oficial interrumpido; las semillas terminadas quedan guardadas")
        raise
    except Exception as exc:
        monitor_test.registrar(f"ERROR test oficial {type(exc).__name__}: {exc}")
        monitor_test.fallar(f"Test oficial con error {type(exc).__name__}; consultar monitor.log")
        raise
    monitor_test.finalizar("Test oficial completado (5 semillas por variante)")
    df_test_final = pd.DataFrame(datos_test)
    resumen = (
        df_test_final.groupby("variante")
        .agg(
            f1_mean=("f1_macro", "mean"),
            f1_std=("f1_macro", "std"),
            f1_sin_coincidencias_mean=("f1_macro_sin_coincidencias", "mean"),
            f1_sin_coincidencias_std=("f1_macro_sin_coincidencias", "std"),
            n_coincidencias=("n_coincidencias", "first"),
        )
        .reset_index()
    )
    display(resumen)
    probabilidades_test = pd.concat(
        [
            pd.read_csv(FIT_STATE_DIR / f"{v}_test_seed{s}_predicciones.csv", dtype={"ID": str})
            for v in VARIANTES
            for s in SEMILLAS_FINALES
        ],
        ignore_index=True,
    )
    csv_atomico(probabilidades_test, RESULTS_DIR / "distilbeto_test_probabilidades.csv")
    print(
        "Probabilidades del test guardadas en:", RESULTS_DIR / "distilbeto_test_probabilidades.csv"
    )
    csv_atomico(resumen, RESULTS_DIR / "distilbeto_test_resumen.csv")
else:
    print("TEST OFICIAL BLOQUEADO; no se han cargado archivos de test.")

## 18. Resumen y lista de verificación

1. Ejecutar primero `01_preprocesamiento_dl_transformers.ipynb`.
2. Confirmar que `manifest_preprocesamiento_transformers.csv` existe (versión `transformers_comun_v2`) y que `MESSAGE_TRANSFORMER` contiene texto.
3. Opcional: una corrida corta con `MODO="prueba"` para comprobar el entorno (la optimización final y el test se omiten).
4. Verificar `PRE-FLIGHT DE DATOS: OK`.
5. DistilBETO no retira registros del train en función del test.
6. `GROUP_ID` debe construirse con el texto original, cada representación de entrada y su secuencia tokenizada.
7. Los grupos con etiquetas mixtas se documentan, pero no se separan entre folds.
8. Para la corrida definitiva, reiniciar kernel, usar `MODO="completo"` y confirmar **5 Outer × 3 Inner × 20 trials**.
9. Nested CV y optimización final corren seguidas (solo train). El test oficial queda bloqueado hasta poner `EJECUTAR_TEST_OFICIAL=True` y `CONFIRMACION_TEST="TEST_OFICIAL_CONGELADO"`.
10. Observar el panel de progreso y `monitor.log`; los checkpoints permiten reanudar una interrupción.
11. Al finalizar, recopilar resultados por Outer Fold, resumen Nested CV, trials de Optuna y curvas.

Con el modo completo, el máximo teórico es:

`3 × 5 × (20 × 3 + 1) = 915 fine-tunings`.